In [1]:
"""
ПРОГРАММА ДЛЯ РАСЧЁТА ОБЪЁМА ВЫБОРКИ
=====================================
Три метода определения необходимого количества уголовных дел для исследования

Автор: Андриянов А  
Основано на: методологии расчёта репрезентативной выборки
"""

import numpy as np
import pandas as pd
from scipy import stats
import math


In [2]:
# -*- coding: utf-8 -*-
# ============================================================================
# МЕТОД 1: КОЛИЧЕСТВЕННЫЙ ПРИЗНАК (возраст, срок, сумма ущерба)
# ============================================================================

def method_1_quantitative_trait(pilot_data, N_population, confidence_level=0.95, margin_error=1):
    """
    Расчёт объёма выборки для количественного признака (например, возраст)
    
    Смысл метода: когда у нас есть числовые значения (годы, месяцы, рубли),
    мы используем дисперсию этих значений из пилотного исследования.
    
    Parameters:
    -----------
    pilot_data : list - пилотные данные (например, список возрастов)
    N_population : int - объём генеральной совокупности
    confidence_level : float - доверительная вероятность (0.95 = 95%)
    margin_error : float - допустимая ошибка (в тех же единицах, что и данные)
    
    Returns:
    --------
    dict - результаты расчёта
    """
    print("\n" + "="*70)
    print("МЕТОД 1: РАСЧЁТ ПО КОЛИЧЕСТВЕННОМУ ПРИЗНАКУ")
    print("="*70)
    
    # Преобразуем в pandas Series для удобства
    data = pd.Series(pilot_data)
    
    # Основные статистики по пилотной выборке
    n_pilot = len(data)
    mean_value = data.mean()
    variance = data.var()
    std_dev = data.std()
    
    print(f"\n📊 ПИЛОТНЫЕ ДАННЫЕ (n = {n_pilot} наблюдений):")
    print(f"   Среднее значение: {mean_value:.2f}")
    print(f"   Дисперсия (σ²): {variance:.2f}")
    print(f"   Стандартное отклонение (σ): {std_dev:.2f}")
    
    # Критерий Стьюдента (двусторонний)
    alpha = 1 - confidence_level
    t_value = stats.t.ppf(1 - alpha/2, df=N_population-1)
    
    print(f"\n📐 ПАРАМЕТРЫ ИССЛЕДОВАНИЯ:")
    print(f"   Генеральная совокупность N = {N_population}")
    print(f"   Доверительная вероятность: {confidence_level*100}%")
    print(f"   Уровень значимости (α) = {alpha:.3f}")
    print(f"   Критерий Стьюдента (t) = {t_value:.4f}")
    print(f"   Предельная ошибка (Δ) = {margin_error}")
    
    # Расчёт объёма выборки
    # Формула для бесповторного отбора
    numerator = (t_value**2) * variance * N_population
    denominator = (margin_error**2) * N_population + (t_value**2) * variance
    sample_size = numerator / denominator
    
    # Округляем вверх (всегда с запасом)
    sample_size = int(math.ceil(sample_size))
    sample_percent = (sample_size / N_population) * 100
    
    print(f"\n✅ РЕЗУЛЬТАТ МЕТОДА 1:")
    print(f"   Необходимый объём выборки: n = {sample_size} единиц")
    print(f"   Это {sample_percent:.1f}% от генеральной совокупности")
    print(f"   С такой выборкой среднее значение будет определено")
    print(f"   с точностью ±{margin_error} с вероятностью {confidence_level*100}%")
    
    return {
        'method': 'Количественный признак',
        'sample_size': sample_size,
        'sample_percent': sample_percent,
        'parameters': {
            'N': N_population,
            't': t_value,
            'variance': variance,
            'delta': margin_error
        }
    }


# ============================================================================
# МЕТОД 2: СТРАТИФИЦИРОВАННАЯ ВЫБОРКА (учёт групп в совокупности)
# ============================================================================

def method_2_stratified_by_groups(N_total, group_sizes, group_proportions_with_trait, 
                                   confidence_level=0.95, margin_error=0.05):
    """
    Расчёт объёма стратифицированной выборки для оценки доли признака
    
    Смысл метода: генеральная совокупность разделена на группы (страты),
    и мы знаем (или предполагаем) долю интересующего признака ВНУТРИ каждой группы.
    
    Parameters:
    -----------
    N_total : int - общий размер генеральной совокупности
    group_sizes : list - размеры каждой группы
    group_proportions_with_trait : list - доля признака ВНУТРИ каждой группы
    confidence_level : float - доверительная вероятность
    margin_error : float - предельная ошибка (в долях, 0.05 = 5%)
    
    Returns:
    --------
    dict - результаты расчёта
    """
    print("\n" + "="*70)
    print("МЕТОД 2: СТРАТИФИЦИРОВАННАЯ ВЫБОРКА (ПО ГРУППАМ)")
    print("="*70)
    
    group_sizes = np.array(group_sizes)
    group_props = np.array(group_proportions_with_trait)
    
    # Проверка сумм
    if not np.isclose(sum(group_sizes), N_total):
        print(f"⚠️ ВНИМАНИЕ: сумма размеров групп ({sum(group_sizes)})")
        print(f"   не равна общей совокупности ({N_total})")
        # Нормализуем
        group_sizes = group_sizes / sum(group_sizes) * N_total
        group_sizes = np.round(group_sizes).astype(int)
    
    # Доли групп в совокупности
    group_weights = group_sizes / N_total
    
    # Внутригрупповые дисперсии для качественного признака (p_i * q_i)
    within_group_variances = group_props * (1 - group_props)
    
    # Средняя внутригрупповая дисперсия (взвешенная)
    sigma2_bar = np.sum(within_group_variances * group_weights)
    
    print(f"\n📊 СТРУКТУРА ГРУПП:")
    for i, (size, prop, weight) in enumerate(zip(group_sizes, group_props, group_weights)):
        q = 1 - prop
        print(f"   Группа {i+1}: размер = {size} ({weight*100:.1f}%)")
        print(f"      p = {prop:.3f}, q = {q:.3f}")
        print(f"      внутригрупповая дисперсия = {prop*q:.4f}")
    
    print(f"\n📐 ПАРАМЕТРЫ ИССЛЕДОВАНИЯ:")
    print(f"   Генеральная совокупность N = {N_total}")
    print(f"   Доверительная вероятность: {confidence_level*100}%")
    print(f"   Предельная ошибка (Δ) = {margin_error*100}%")
    print(f"   Средняя внутригрупповая дисперсия (σ²̄) = {sigma2_bar:.4f}")
    
    # Критерий из нормального распределения (для больших выборок)
    alpha = 1 - confidence_level
    z_value = stats.norm.ppf(1 - alpha/2)
    print(f"   Критерий (z) = {z_value:.4f}")
    
    # Расчёт объёма выборки
    numerator = z_value**2 * sigma2_bar
    denominator = margin_error**2 + (z_value**2 * sigma2_bar) / N_total
    sample_size = numerator / denominator
    
    sample_size = int(math.ceil(sample_size))
    sample_percent = (sample_size / N_total) * 100
    
    print(f"\n✅ РЕЗУЛЬТАТ МЕТОДА 2:")
    print(f"   Необходимый объём выборки: n = {sample_size} единиц")
    print(f"   Это {sample_percent:.1f}% от генеральной совокупности")
    print(f"\n   📌 ВАЖНО: выборка должна быть СТРАТИФИЦИРОВАННОЙ:")
    print(f"      Пропорционально распределению по группам:")
    for i, (size, weight) in enumerate(zip(group_sizes, group_weights)):
        n_group = int(round(sample_size * weight))
        print(f"      Группа {i+1}: отобрать {n_group} ед. ({weight*100:.1f}% от выборки)")
    
    return {
        'method': 'Стратифицированная выборка',
        'sample_size': sample_size,
        'sample_percent': sample_percent,
        'group_allocation': [int(round(sample_size * w)) for w in group_weights],
        'parameters': {
            'N': N_total,
            'z': z_value,
            'sigma2_bar': sigma2_bar,
            'delta': margin_error
        }
    }


# ============================================================================
# МЕТОД 3: ГРАДАЦИИ ПРИЗНАКА (категориальный признак с несколькими значениями)
# ============================================================================

def method_3_categorical_trait(category_frequencies, N_population, 
                                confidence_level=0.95, relative_error=0.07):
    """
    Расчёт объёма выборки для категориального признака с градациями
    
    Смысл метода: признак имеет несколько категорий (например, способы убийств),
    и мы хотим оценить распределение по этим категориям.
    Дисперсия считается по частотам категорий.
    
    Parameters:
    -----------
    category_frequencies : list - частоты по категориям из пилотной выборки
    N_population : int - объём генеральной совокупности
    confidence_level : float - доверительная вероятность
    relative_error : float - относительная ошибка (в долях от среднего)
    
    Returns:
    --------
    dict - результаты расчёта
    """
    print("\n" + "="*70)
    print("МЕТОД 3: КАТЕГОРИАЛЬНЫЙ ПРИЗНАК С ГРАДАЦИЯМИ")
    print("="*70)
    
    freqs = np.array(category_frequencies)
    n_categories = len(freqs)
    total_pilot = sum(freqs)
    
    # Выборочное среднее (средняя частота на категорию)
    mean_freq = total_pilot / n_categories
    
    # Дисперсия частот
    variance = np.var(freqs, ddof=1)  # ddof=1 для несмещённой оценки
    std_dev = np.sqrt(variance)
    
    # Коэффициент вариации (показывает неоднородность)
    cv = std_dev / mean_freq if mean_freq != 0 else float('inf')
    
    print(f"\n📊 ПИЛОТНЫЕ ДАННЫЕ:")
    print(f"   Всего категорий: {n_categories}")
    print(f"   Сумма частот: {total_pilot}")
    print(f"   Распределение: {freqs}")
    print(f"\n📈 СТАТИСТИКИ:")
    print(f"   Средняя частота на категорию (ˉx) = {mean_freq:.2f}")
    print(f"   Дисперсия (σ²) = {variance:.2f}")
    print(f"   Стандартное отклонение (σ) = {std_dev:.2f}")
    print(f"   Коэффициент вариации (CV) = {cv:.2f} ({cv*100:.1f}%)")
    
    if cv > 1:
        print(f"   ⚠️ Очень высокий разброс (CV > 100%) — данные крайне неоднородны")
    
    # Предельная ошибка в абсолютных единицах
    delta_abs = relative_error * mean_freq
    
    print(f"\n📐 ПАРАМЕТРЫ ИССЛЕДОВАНИЯ:")
    print(f"   Генеральная совокупность N = {N_population}")
    print(f"   Доверительная вероятность: {confidence_level*100}%")
    print(f"   Относительная ошибка: {relative_error*100}%")
    print(f"   Абсолютная ошибка (Δ) = {delta_abs:.2f} преступления на категорию")
    
    # Критерий Стьюдента
    alpha = 1 - confidence_level
    t_value = stats.t.ppf(1 - alpha/2, df=N_population-1)
    print(f"   Критерий Стьюдента (t) = {t_value:.4f}")
    
    # Расчёт объёма выборки
    numerator = (t_value**2) * variance * N_population
    denominator = (delta_abs**2) * N_population + (t_value**2) * variance
    sample_size = numerator / denominator
    
    sample_size = int(math.ceil(sample_size))
    sample_percent = (sample_size / N_population) * 100
    
    print(f"\n✅ РЕЗУЛЬТАТ МЕТОДА 3:")
    print(f"   Необходимый объём выборки: n = {sample_size} единиц")
    print(f"   Это {sample_percent:.1f}% от генеральной совокупности")
    print(f"   С такой выборкой средняя частота на категорию")
    print(f"   будет определена с точностью ±{delta_abs:.2f} (±{relative_error*100}%)")
    print(f"   с вероятностью {confidence_level*100}%")
    
    return {
        'method': 'Категориальный признак',
        'sample_size': sample_size,
        'sample_percent': sample_percent,
        'parameters': {
            'N': N_population,
            't': t_value,
            'variance': variance,
            'mean': mean_freq,
            'delta_abs': delta_abs,
            'delta_rel': relative_error
        }
    }


# ============================================================================
# ПРИМЕРЫ ИСПОЛЬЗОВАНИЯ (ТЕСТИРОВАНИЕ НА ВАШИХ ДАННЫХ)
# ============================================================================

if __name__ == "__main__":
    
    print("\n" + "★"*70)
    print("ПРОГРАММА РАСЧЁТА ОБЪЁМА ВЫБОРКИ ДЛЯ КРИМИНОЛОГИЧЕСКИХ ИССЛЕДОВАНИЙ")
    print("★"*70)
    
    # ------------------------------------------------------------
    # ПРИМЕР 1: Ваши данные из МЕТОДА 1 (возраст)
    # ------------------------------------------------------------
    pilot_ages = [22, 25, 27, 31, 34, 34, 36, 38, 39, 41, 42, 43, 45, 46, 
                  47, 48, 49, 51, 53, 55, 38, 29, 33, 44, 52]
    
    result1 = method_1_quantitative_trait(
        pilot_data=pilot_ages,
        N_population=245,
        confidence_level=0.95,
        margin_error=1
    )
    
    # ------------------------------------------------------------
    # ПРИМЕР 2: Ваши данные из МЕТОДА 2 (пол)
    # ------------------------------------------------------------
    # Группы: мужчины (90.28% от 1000), женщины (9.72% от 1000)
    # Предполагаемые доли признака (например, "наличие судимости") внутри групп
    # (это гипотетические значения для демонстрации)
    
    result2 = method_2_stratified_by_groups(
        N_total=1000,
        group_sizes=[903, 97],
        group_proportions_with_trait=[0.80, 0.30],  # 80% мужчин, 30% женщин
        confidence_level=0.95,
        margin_error=0.05  # ошибка 5%
    )
    
    # ------------------------------------------------------------
    # ПРИМЕР 3: Ваши данные из МЕТОДА 3 (способы убийств)
    # ------------------------------------------------------------
    # Частоты из пилотной выборки (287 дел)
    crime_methods = [163, 57, 47, 18, 2]  # колюще-режущее, тупые, огнестрел, иные, взрывные
    
    result3 = method_3_categorical_trait(
        category_frequencies=crime_methods,
        N_population=1000,
        confidence_level=0.95,
        relative_error=0.068  # 6.8% (ваше значение Δ/ˉx = 3.92/57.4)
    )
    
    # ------------------------------------------------------------
    # СВОДНЫЙ ОТЧЁТ
    # ------------------------------------------------------------
    print("\n" + "="*70)
    print("СВОДНЫЙ ОТЧЁТ ПО ВСЕМ МЕТОДАМ")
    print("="*70)
    
    results = [result1, result2, result3]
    
    for i, res in enumerate(results, 1):
        print(f"\n{i}. {res['method']}:")
        print(f"   → n = {res['sample_size']} ед. ({res['sample_percent']:.1f}% от N)")
    
    print("\n" + "★"*70)
    print("ИТОГОВЫЙ ВЫВОД:")
    print("★"*70)
    print("""
    ТРИ МЕТОДА — ТРИ РАЗНЫХ ВЗГЛЯДА НА ДАННЫЕ:
    
    🔹 МЕТОД 1 (количественный признак) отвечает на вопрос:
       "Сколько дел изучить, чтобы узнать средний возраст?"
       → Используется, когда есть числовые значения
    
    🔹 МЕТОД 2 (стратифицированный) отвечает на вопрос:
       "Сколько дел изучить, чтобы оценить долю признака
        в группах (пол, национальность, категория дел)?"
       → Используется, когда совокупность неоднородна и разделена на группы
    
    🔹 МЕТОД 3 (категориальный) отвечает на вопрос:
       "Сколько дел изучить, чтобы понять структуру распределения
        по способам совершения преступления?"
       → Используется для анализа частот по категориям
    
    📌 ГЛАВНЫЙ СМЫСЛ ПРОГРАММЫ:
    
    Вы НЕ ДОЛЖНЫ изучать все дела подряд — это расточительно.
    Вы НЕ ДОЛЖНЫ брать "на глаз" 100 или 200 дел — это ненаучно.
    
    Программа даёт МАТЕМАТИЧЕСКИ ОБОСНОВАННЫЙ ответ:
    "Возьми ровно СТОЛЬКО дел, отобранных ТАКИМ способом,
     чтобы с вероятностью 95% твои выводы отличались от реальности
     не больше, чем на ЗАДАННУЮ ТОБОЙ ОШИБКУ".
    
    В вашем случае с убийствами:
    • Если нужен средний возраст → 135 дел
    • Если нужна доля по группам (пол) → от 192 до 704 дел (зависит от точности)
    • Если нужна структура по способам → 500 дел
    
    Программа позволяет ОПТИМИЗИРОВАТЬ ресурсы исследования,
    сохраняя НАУЧНУЮ ДОСТОВЕРНОСТЬ результатов.
    """)


★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
ПРОГРАММА РАСЧЁТА ОБЪЁМА ВЫБОРКИ ДЛЯ КРИМИНОЛОГИЧЕСКИХ ИССЛЕДОВАНИЙ
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

МЕТОД 1: РАСЧЁТ ПО КОЛИЧЕСТВЕННОМУ ПРИЗНАКУ

📊 ПИЛОТНЫЕ ДАННЫЕ (n = 25 наблюдений):
   Среднее значение: 40.08
   Дисперсия (σ²): 83.74
   Стандартное отклонение (σ): 9.15

📐 ПАРАМЕТРЫ ИССЛЕДОВАНИЯ:
   Генеральная совокупность N = 245
   Доверительная вероятность: 95.0%
   Уровень значимости (α) = 0.050
   Критерий Стьюдента (t) = 1.9697
   Предельная ошибка (Δ) = 1

✅ РЕЗУЛЬТАТ МЕТОДА 1:
   Необходимый объём выборки: n = 140 единиц
   Это 57.1% от генеральной совокупности
   С такой выборкой среднее значение будет определено
   с точностью ±1 с вероятностью 95.0%

МЕТОД 2: СТРАТИФИЦИРОВАННАЯ ВЫБОРКА (ПО ГРУППАМ)

📊 СТРУКТУРА ГРУПП:
   Группа 1: размер = 903 (90.3%)
      p = 0.800, q = 0.200
      внутригрупповая дисперсия = 0.1600
   Группа 2: размер = 97 (9.7%)
      p = 0.3